In [1]:
from pathlib import Path
from google.colab import files
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib

ruta = Path('dataset_modelo_pm25_24h.csv')
if not ruta.exists():
    files.upload()
d = pd.read_csv(ruta, parse_dates=['fecha_hora'])
# Recuperar solo observaciones disponibles al emitir el pronóstico.
x = pd.DataFrame(index=d.index)
for col in d.filter(regex='_actual$'):
    marca = col.replace('_actual', '_imputado')
    x[col] = d[col].mask(d[marca].eq(1))
x['hora'] = d.fecha_hora.dt.hour
x['mes'] = d.fecha_hora.dt.month
x = x.join(pd.get_dummies(d.estacion, prefix='estacion'))
y = d['PM2.5_objetivo_24h']
fin = d.fecha_hora + pd.Timedelta(hours=24)
entrenar = fin < '2024-01-01'
modelo = HistGradientBoostingRegressor(random_state=42, early_stopping=False)
modelo.fit(x[entrenar], y[entrenar])
joblib.dump(modelo, 'air_byte_pm25_24h.joblib')
for anio in [2024, 2025]:
    evaluar = (d.fecha_hora.dt.year == anio) & (fin.dt.year == anio) & x['PM2.5_actual'].notna()
    for nombre, pred in [('Modelo', modelo.predict(x[evaluar])), ('Persistencia', x.loc[evaluar, 'PM2.5_actual'])]:
        print(anio, nombre, 'MAE', round(mean_absolute_error(y[evaluar], pred), 2), 'RMSE', round(mean_squared_error(y[evaluar], pred)**0.5, 2))

Saving dataset_modelo_pm25_24h.csv to dataset_modelo_pm25_24h.csv
2024 Modelo MAE 7.77 RMSE 10.32
2024 Persistencia MAE 8.84 RMSE 12.51
2025 Modelo MAE 7.29 RMSE 9.8
2025 Persistencia MAE 8.19 RMSE 12.27


In [3]:
# Comparación con historial observado y evaluación de episodios altos
from sklearn.base import clone
import numpy as np

# Recuperar lecturas reales a partir de las etiquetas y mediciones actuales.
claves = ['estacion', 'fecha_hora']
observado = d.loc[d['PM2.5_imputado'].eq(0), claves + ['PM2.5_actual']].set_index(claves)['PM2.5_actual']
posterior = d[claves].assign(fecha_hora=fin, pm=y).set_index(claves)['pm']
serie = observado.combine_first(posterior)
x_hist = x.copy()
for h in [1, 3, 6, 24]:
    pasado = d[claves].assign(fecha_hora=d.fecha_hora - pd.Timedelta(hours=h))
    x_hist[f'pm25_hace_{h}h'] = serie.reindex(pd.MultiIndex.from_frame(pasado)).to_numpy()
con_historial = clone(modelo).fit(x_hist[entrenar], y[entrenar])
# Pico = concentración por encima del percentil 90 histórico de cada estación.
umbral = y[entrenar].groupby(d.loc[entrenar, 'estacion']).quantile(.90)
for anio in (2024, 2025):
    mask = (d.fecha_hora.dt.year == anio) & (fin.dt.year == anio) & x['PM2.5_actual'].notna()
    tabla = pd.DataFrame({'estacion':d.loc[mask,'estacion'], 'real':y[mask], 'base':modelo.predict(x[mask]), 'hist':con_historial.predict(x_hist[mask])})
    def resumen(g):
        p = g[g.real > g.estacion.map(umbral)]
        return [len(g), (g.base-g.real).abs().mean(), (g['hist']-g.real).abs().mean(), len(p), (p.base-p.real).abs().mean(), (p['hist']-p.real).abs().mean(), (p['hist']-p.real).mean()]
    grupos = [('Total',tabla), *tabla.groupby('estacion')]
    print(anio)
    print(pd.DataFrame([resumen(g) for _,g in grupos],index=[k for k,_ in grupos],columns=['N','MAE_base','MAE_hist','Picos','Pico_base','Pico_hist','Sesgo_hist']).round(2).to_string())
joblib.dump(con_historial,'air_byte_pm25_24h_historial.joblib')

2024
                        N  MAE_base  MAE_hist  Picos  Pico_base  Pico_hist  Sesgo_hist
Total               39105      7.77      7.63   3296      18.62      18.32      -18.10
Carvajal-Sevillana   7878      9.96      9.83    638      25.75      25.18      -24.92
Kennedy              7578      7.73      7.42    406      18.29      18.55      -18.42
Puente Aranda        8030      7.36      7.31    671      16.93      16.83      -16.54
Tunal                7670      7.50      7.40    695      20.06      19.45      -19.42
Usaquen              7949      6.31      6.21    886      13.78      13.52      -13.19
2025
                        N  MAE_base  MAE_hist  Picos  Pico_base  Pico_hist  Sesgo_hist
Total               28527      7.29      7.12   1093      22.10      22.44      -22.41
Carvajal-Sevillana   6694     10.54     10.40    418      27.61      27.99      -27.95
Kennedy               904      7.81      7.52     35      17.73      18.85      -18.85
Puente Aranda        7123      6.

['air_byte_pm25_24h_historial.joblib']